# HotpotQA SDPO training in Colab

This notebook uses the refactored plain-language training pipeline: the student creates a full RLM attempt, token selection chooses the requested semantic regions, the hosted Qwen judge supplies feedback, SDPO calculates the loss, and the saved attempt records make every stage inspectable.

Before running: select a GPU under **Runtime → Change runtime type** and add `HF_TOKEN` in Colab Secrets with **Make calls to Inference Providers** permission. Add `GITHUB_TOKEN` only if the repository is private. Run the cells in order.

In [ ]:
import os
import sys

import torch
from google.colab import drive, userdata

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["PYTHONUNBUFFERED"] = "1"

drive.mount("/content/drive")

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN is missing or notebook access is disabled")
os.environ["HF_TOKEN"] = hf_token

# GITHUB_TOKEN is optional and is never placed in a command or printed.
try:
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = None

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected; change the Colab runtime type to GPU")

print("Python:", sys.version.split()[0])
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())
print("Hugging Face token loaded; secret value was not printed.")

## Prepare the repository

This cell reuses `/content/rlm-ib` when it is already a Git checkout. It never deletes the directory and never includes a token in a Git command or traceback.

In [ ]:
import base64
import subprocess
from pathlib import Path

REPOSITORY_URL = "https://github.com/mmzinn12/rlm-ib.git"
REPOSITORY_PATH = Path("/content/rlm-ib")
BRANCH = "sdpo"

git_environment = os.environ.copy()
if github_token:
    authorization = base64.b64encode(
        f"x-access-token:{github_token}".encode()
    ).decode()
    git_environment["GIT_CONFIG_COUNT"] = "1"
    git_environment["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
    git_environment["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {authorization}"

def run_git(arguments):
    result = subprocess.run(
        arguments,
        env=git_environment,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or "Git command failed")
    return result.stdout.strip()

if REPOSITORY_PATH.exists():
    if not (REPOSITORY_PATH / ".git").is_dir():
        raise RuntimeError(
            f"{REPOSITORY_PATH} exists but is not a Git checkout; restart the runtime"
        )
    run_git(["git", "-C", str(REPOSITORY_PATH), "fetch", "origin", BRANCH])
    local_branch = subprocess.run(
        ["git", "-C", str(REPOSITORY_PATH), "show-ref", "--verify", f"refs/heads/{BRANCH}"],
        env=git_environment,
        capture_output=True,
        text=True,
    )
    if local_branch.returncode == 0:
        run_git(["git", "-C", str(REPOSITORY_PATH), "switch", BRANCH])
    else:
        run_git(["git", "-C", str(REPOSITORY_PATH), "switch", "--track", "-c", BRANCH, f"origin/{BRANCH}"])
    run_git(["git", "-C", str(REPOSITORY_PATH), "pull", "--ff-only", "origin", BRANCH])
else:
    run_git([
        "git", "clone", "--branch", BRANCH, "--single-branch",
        REPOSITORY_URL, str(REPOSITORY_PATH),
    ])

print("Branch:", run_git(["git", "-C", str(REPOSITORY_PATH), "branch", "--show-current"]))
print("Commit:", run_git(["git", "-C", str(REPOSITORY_PATH), "rev-parse", "--short", "HEAD"]))
%cd /content/rlm-ib

In [ ]:
%pip install -q -e /content/rlm-ib -e "/content/rlm-ib/training[colab,hub-datasets]"

In [ ]:
import importlib.metadata as metadata
import subprocess

from rlm_train.gateways.colab import validate_colab_device

print("RLM:", metadata.version("rlms"))
print("RLM Train:", metadata.version("rlm-train"))
print("OpenAI client:", metadata.version("openai"))
print(validate_colab_device())

tests = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_plain_language_training.py",
        "tests/test_hotpotqa_dataset.py",
        "tests/test_judge_modes.py",
        "tests/test_render.py",
    ],
    cwd="/content/rlm-ib/training",
    capture_output=True,
    text=True,
)
print(tests.stdout)
if tests.returncode != 0:
    print(tests.stderr)
    raise RuntimeError("Refactored training-package checks failed")

## Verify the hosted Qwen judge

The judge uses the OpenAI-compatible protocol exposed by the Hugging Face router. `provider="openai"` in the run settings names that protocol adapter; the actual model is Qwen and billing/authentication remain with Hugging Face. This small request verifies authentication, routing, the Responses API, and strict structured output before training begins.

In [ ]:
from openai import OpenAI

JUDGE_BASE_URL = "https://router.huggingface.co/v1"
JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct:together"

judge_client = OpenAI(
    base_url=JUDGE_BASE_URL,
    api_key=os.environ["HF_TOKEN"],
)
judge_probe = judge_client.responses.create(
    model=JUDGE_MODEL,
    instructions="Return the requested JSON health-check result.",
    input="Confirm that the judge endpoint works.",
    text={
        "format": {
            "type": "json_schema",
            "name": "judge_health_check",
            "schema": {
                "type": "object",
                "properties": {
                    "status": {"type": "string", "enum": ["ok"]}
                },
                "required": ["status"],
                "additionalProperties": False,
            },
            "strict": True,
        }
    },
)
print("Judge response:", judge_probe.output_text)

## Configure one canonical training run

The controls at the top of the next cell are the values you will usually change. The first run is deliberately bounded: five possible HotpotQA records, one optimizer step, one attempt at a time, and locally stored artifacts. `helper_questions` activates the dedicated semantic token-selection path.

In [ ]:
from datetime import UTC, datetime
from pathlib import Path

from rlm_train import RunSpec

# ---- Common experiment controls ----
STUDENT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
TRAIN_MAX_RECORDS = 5
EVALUATION_MAX_RECORDS = 2
MAX_OPTIMIZER_STEPS = 1
TOKEN_SCOPE = "helper_questions"
SAVE_FINAL_CHECKPOINT = False

RUN_NAME = f"hotpotqa-sdpo-{datetime.now(UTC):%Y%m%dT%H%M%SZ}"
OUTPUT_DIRECTORY = Path("/content/rlm-ib-outputs") / RUN_NAME
CONFIG_DIRECTORY = Path("/content/drive/MyDrive/rlm-ib-configs")
CONFIG_DIRECTORY.mkdir(parents=True, exist_ok=True)
RUN_SPEC_PATH = CONFIG_DIRECTORY / f"{RUN_NAME}.json"
PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "fp16"

prompt_directory = Path("/content/rlm-ib/training/prompts")
system_prompt_text = "\n\n".join(
    [
        (prompt_directory / "rlm_root_system_prompt.md").read_text(),
        (prompt_directory / "rlm_decomposition_fewshot.md").read_text(),
    ]
)
system_prompt = system_prompt_text.replace("{", "{{").replace("}", "}}")

run_spec = RunSpec.model_validate(
    {
        "schema_version": 1,
        "student": {
            "adapter": "transformers",
            "model_id": STUDENT_MODEL,
            "policy_owner": "student",
            "trainable": True,
            "generation": {
                "max_prompt_tokens": 4096,
                "max_new_tokens": 256,
                "temperature": 0.8,
                "top_p": 0.95,
                "do_sample": True,
                "use_chat_template": True,
                "allow_prompt_truncation": False,
            },
        },
        "rollout": {
            "engine": "rlm",
            "environment": "local",
            "max_depth": 2,
            "max_iterations": 6,
            "max_concurrent_subcalls": 1,
            "system_prompt": system_prompt,
        },
        "judge": {
            "provider": "openai",
            "model": JUDGE_MODEL,
            "model_revision": "qwen2.5-7b-together-route-v1",
            "schema": "edge-information-v1",
            "prompt_version": "edge-information-v1",
            "mode": "categorical",
            "api_key_environment": "HF_TOKEN",
            "base_url": JUDGE_BASE_URL,
            "max_attempts": 3,
            "cache_path": str(OUTPUT_DIRECTORY / "judge-cache.sqlite"),
        },
        "teacher": {"strategy": "current_policy"},
        "feedback": {
            "default_scope": "retrospective_local",
            "upstream_depth": 1,
            "downstream_depth": 1,
            "include_siblings": False,
            "allow_privileged_hindsight_distillation": False,
        },
        "objectives": {
            "sdpo": {
                "enabled": True,
                "weight": 1.0,
                "token_scope": TOKEN_SCOPE,
                "feedback_scope": "retrospective_local",
                "divergence": "reverse_kl",
                "target_support": "top_k_with_tail",
                "top_k": 32,
            }
        },
        "training_dataset": {
            "adapter": "hotpotqa",
            "source": "hotpotqa/hotpot_qa",
            "subset": "distractor",
            "split": "train",
            "max_records": TRAIN_MAX_RECORDS,
            "name": "hotpotqa",
            "version": "distractor",
        },
        "evaluation_datasets": [
            {
                "adapter": "hotpotqa",
                "source": "hotpotqa/hotpot_qa",
                "subset": "distractor",
                "split": "validation",
                "max_records": EVALUATION_MAX_RECORDS,
                "name": "hotpotqa-validation",
                "version": "distractor",
            }
        ],
        "evaluation": {
            "recursive_policy": True,
            "samples_per_problem": 1,
            "base_seed": 0,
        },
        "artifacts": {
            "output_directory": str(OUTPUT_DIRECTORY),
            "rollout_json": "all",
            "metrics_jsonl": True,
            "checkpoint_interval": None,
            "retain_checkpoints": 1,
            "save_final_checkpoint": SAVE_FINAL_CHECKPOINT,
        },
        "runtime": {
            "device": "cuda",
            "precision": PRECISION,
            "seed": 0,
            "gradient_accumulation_steps": 1,
            "max_optimizer_steps": MAX_OPTIMIZER_STEPS,
            "learning_rate": 5e-5,
            "max_gradient_norm": 1.0,
            "warmup_steps": 0,
            "scheduler": "linear",
        },
    }
)

run_spec.write_resolved(RUN_SPEC_PATH)
run_spec = RunSpec.from_file(RUN_SPEC_PATH)

print("Run spec:", RUN_SPEC_PATH)
print("Output directory:", OUTPUT_DIRECTORY)
print("Student:", run_spec.student.model_id)
print("Judge:", run_spec.judge.model)
print("Selected text:", run_spec.objectives.sdpo.token_scope.value)
print("Precision:", run_spec.runtime.precision)

In [ ]:
from rlm_train.datasets import build_dataset

training_records = build_dataset(run_spec.training_dataset).records()
first_record = training_records[0]

print("Loaded records:", len(training_records))
print("Record ID:", first_record.record_id)
print("Question:", first_record.public_task["question"])
print("Context preview:")
print(first_record.public_task["context"][:800])
print("Metadata:", first_record.metadata)

## Train

This calls the same public API as the CLI. It creates the student, full-RLM attempt runner, feedback collector, semantic token selection, SDPO calculation, optimizer, and saved-run writers from the single validated run specification.

In [ ]:
from rlm_train.gateways.colab import train

training_result = train(run_spec, verbose=True)
training_result

## Inspect raw rollouts, selected tokens, and judge feedback

This reads the durable attempt files produced by training. The recursion tree is a compact overview; the following sections show the exact sampled text, the semantic ranges selected for SDPO, and the complete structured feedback returned by Qwen.

In [ ]:
import json

from IPython.display import Markdown, display
from rlm_train.trajectory.render import render_recursion_tree
from rlm_train.trajectory.replay import load_annotated_rollout

attempt_paths = sorted((OUTPUT_DIRECTORY / "attempts").glob("*.json"))
if not attempt_paths:
    raise FileNotFoundError(f"No saved attempts found under {OUTPUT_DIRECTORY / 'attempts'}")

print(f"Found {len(attempt_paths)} saved attempt(s).")

for attempt_path in attempt_paths:
    rollout = load_annotated_rollout(attempt_path)
    display(Markdown(f"# Rollout `{rollout.rollout_id}`"))

    display(Markdown("## Recursion tree"))
    print(render_recursion_tree(rollout, max_text_chars=500))

    display(Markdown("## Raw student generations"))
    for index, generation in enumerate(rollout.annotations.generations, start=1):
        print("=" * 80)
        print(f"Generation {index}: {generation.generation_id}")
        print("Execution node:", generation.node_id)
        print("Generated tokens:", len(generation.token_ids))
        print("-" * 80)
        print(generation.text)

    display(Markdown("## Tokens selected for each training method"))
    for method_name, selection in rollout.annotations.objective_selections.items():
        print(method_name, "active tokens:", selection.active_token_count)
        for selected_range in selection.ranges:
            print(
                " ", selected_range.generation_id,
                f"tokens {selected_range.token_start}:{selected_range.token_end}",
                "reason:", selected_range.reason,
            )

    display(Markdown("## Final submitted answer"))
    print(rollout.result.get("final_answer", "No final answer was submitted."))

    display(Markdown("## Raw Qwen judge feedback"))
    if not rollout.feedback.judge_assessments:
        print("No judge assessment was produced; this rollout created no judgeable helper edge.")
    for index, assessment in enumerate(rollout.feedback.judge_assessments, start=1):
        print("=" * 80)
        print("Assessment:", index)
        print("Provider:", assessment.get("provider"))
        print("Evaluated edges:", assessment.get("focal_edge_ids"))
        print(json.dumps(assessment.get("content", {}), indent=2, ensure_ascii=False))

## Inspect and optionally preserve artifacts

The run stays on fast ephemeral Colab storage during training. The last cell copies it to a uniquely named Google Drive directory after completion.

In [ ]:
import json
import shutil

metrics_path = OUTPUT_DIRECTORY / "metrics.jsonl"
if metrics_path.exists():
    print("Metrics:")
    for line in metrics_path.read_text().splitlines():
        print(json.dumps(json.loads(line), indent=2))

print("\nArtifacts:")
for path in sorted(OUTPUT_DIRECTORY.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIRECTORY))

drive_destination = Path("/content/drive/MyDrive/rlm-ib-runs") / RUN_NAME
drive_destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(OUTPUT_DIRECTORY, drive_destination)
print("\nCopied run to:", drive_destination)